# Analyse: Helligkeit und Kontrast


In [3]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
from tqdm import tqdm

In [ ]:
DATA_DIR =  Path('../../data')

COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_brightness_contrast.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'  # one folder with per-video subfolders
SOURCE_FILTER = None  
DEFAULT_MAX_FRAMES_PER_VIDEO = 60  #Fallback

device = torch.device("mps") if torch.backends.mps.is_available() else 'cpu'
print(f'Running on device: {device}')


Running on device: cpu


In [ ]:
df = pd.read_csv(COMBINED_CSV)
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'


def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()


video_ids = df[video_id_col].astype(str)
df['has_frames'] = [has_frames(video_id) for video_id in video_ids]

missing_ids = video_ids[~df['has_frames']].tolist()
print(f'Total videos in CSV: {len(df)}')
print(f'Videos with frame folder: {df["has_frames"].sum()}')
print(f'Videos missing frame folder: {len(missing_ids)}')

if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')


Total videos in CSV: 500
Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


In [ ]:
def compute_brightness_contrast(image_path):
    """Berechnet Helligkeit und Kontrast eines Frames mit OpenCV."""
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            return np.nan, np.nan
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness = np.mean(gray)
        contrast = np.std(gray)
        return brightness, contrast
    except Exception as e:
        print(f"?? Error reading {image_path}: {e}")
        return np.nan, np.nan

def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []

    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        return []

    # Ziel: ein Frame pro Videosekunde
    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, len(frame_files)))
    step = max(1, len(frame_files) // target_frames)
    return frame_files[::step][:target_frames]


In [ ]:
# Alle Videos verarbeiten
brightness_values = []
contrast_values = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    video_id = str(row.get('id') or row.get('video_id'))
    duration_seconds = row.get('video_duration_seconds')
    
    frames = sample_frame_paths(video_id, duration_seconds=duration_seconds)
    if not frames:
        brightness_values.append(np.nan)
        contrast_values.append(np.nan)
        continue

    br_list, ct_list = [], []
    for fp in frames:
        br, ct = compute_brightness_contrast(fp)
        if not np.isnan(br):
            br_list.append(br)
            ct_list.append(ct)

    if br_list:
        mean_brightness = np.mean(br_list)
        mean_contrast = np.mean(ct_list)
    else:
        mean_brightness, mean_contrast = np.nan, np.nan

    brightness_values.append(mean_brightness)
    contrast_values.append(mean_contrast)

# Ergebnis-Spalten ergänzen und CSV speichern
df["brightness_index"] = brightness_values
df["contrast_index"] = contrast_values
df.to_csv(OUTPUT_CSV, index=False)
print(f"? Saved results to {OUTPUT_CSV}")


100%|██████████| 500/500 [02:04<00:00,  4.01it/s]

? Saved results to ..\..\data\04_analysis_results\visual_features\01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_brightness_contrast.csv


In [14]:
df[['influencer_type', 'brightness_index', 'contrast_index']].groupby('influencer_type').describe()

brightness_index                                    \
                           count        mean        std        min   
influencer_type                                                      
ai                         250.0  114.907464  34.688726  15.891666   
real                       250.0  120.949318  22.684990  35.873313   

                                                                 \
                        25%         50%         75%         max   
influencer_type                                                   
ai                94.596955  120.803398  139.299015  199.456762   
real             109.713991  122.409226  135.131924  185.425825   

                contrast_index                                              \
                         count       mean        std        min        25%   
influencer_type                                                              
ai                       250.0  59.556350  11.374979  33.659400  51.779454   
real                     250.0  60.026557   7.223060  33.886149  55.669880   

                                                  
                       50%        75%        max  
influencer_type                                   
ai               59.281091  67.100554  87.942020  
real             60.071064  64.218401  88.945214